In [1]:
## Standard libraries
import os
import json
import math
import numpy as np
import time

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgb
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0
import seaborn as sns
sns.reset_orig()
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
# Torchvision
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
# PyTorch Lightning
try:
    import pytorch_lightning as pl
except ModuleNotFoundError: # Google Colab does not have PyTorch Lightning installed by default. Hence, we do it here if necessary
    !pip install --quiet pytorch-lightning>=1.4
    import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

# Setting the seed
pl.seed_everything(42)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


/tmp/ipykernel_12619/3001796416.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('svg', 'pdf') # For export
INFO:lightning_fabric.utilities.seed:Seed set to 42


cpu


## GCNs (Graph Convolutional Netwroks)

GCNs are similar to convolutions in images in the sense that the “filter” parameters are typically shared over all locations in the graph.



*   GCNs rely on message passing methods, which means that vertices exchange information with the neighbors, and send “messages” to each other.

![image.png](https://zoshs2.github.io/assets/img/post/gcn_implement/GCN_message_passing.png)


The first step is that each node creates a feature vector that represents the message it wants to send to all its neighbors. In the second step, the messages are sent to the neighbors, so that a node receives one message per adjacent node.

Here, the usual way to go is to sum or take the mean. Given the previous features of nodes $H^{(l)}$, the GCN layer is defined as follows:

$$
H^{(l+1)} = \sigma \left( \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} H^{(l)} W^{(l)} \right)
$$

$W^{(l)}$ is the weight parameters with which we transform the input features into messages $(H^{(l)} W^{(l)})$.

To the adjacency matrix $A$ we add the identity matrix so that each node sends its own message also to itself:
$$
\hat{A} = A + I
$$

Finally, to take the average instead of summing, we calculate the matrix $\hat{D}$ which is a diagonal matrix with $D_{ii}$ denoting the number of neighbors node $i$ has.

$\sigma$ represents an arbitrary activation function, and not necessarily the sigmoid (usually a ReLU-based activation function is used in GNNs).

In [2]:
class GCNLayer(nn.Module):

    def __init__(self, c_in, c_out):
        super().__init__()
        self.projection = nn.Linear(c_in, c_out)

    def forward(self, node_feats, adj_matrix):
        """
        Inputs:
            node_feats - Tensor with node features of shape [batch_size, num_nodes, c_in]
            adj_matrix - Batch of adjacency matrices of the graph. If there is an edge from i to j, adj_matrix[b,i,j]=1 else 0.
                         Supports directed edges by non-symmetric matrices. Assumes to already have added the identity connections.
                         Shape: [batch_size, num_nodes, num_nodes]
        """
        # Num neighbours = number of incoming edges
        num_neighbours = adj_matrix.sum(dim=-1, keepdims=True)
        node_feats = self.projection(node_feats)
        node_feats = torch.bmm(adj_matrix, node_feats)
        node_feats = node_feats / num_neighbours
        return node_feats

To further understand the GCN layer, we can apply it to our example graph above. First, let’s specify some node features and the adjacency matrix with added self-connections:

In [3]:
node_feats = torch.arange(8, dtype=torch.float32).view(1, 4, 2)
adj_matrix = torch.Tensor([[[1, 1, 0, 0],
                            [1, 1, 1, 1],
                            [0, 1, 1, 1],
                            [0, 1, 1, 1]]])

print("Node features:\n", node_feats)
print("\nAdjacency matrix:\n", adj_matrix)

Node features:
 tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])

Adjacency matrix:
 tensor([[[1., 1., 0., 0.],
         [1., 1., 1., 1.],
         [0., 1., 1., 1.],
         [0., 1., 1., 1.]]])


Next, let’s apply a GCN layer to it.

In [4]:
layer = GCNLayer(c_in=2, c_out=2)
layer.projection.weight.data = torch.Tensor([[1., 0.], [0., 1.]])
layer.projection.bias.data = torch.Tensor([0., 0.])

with torch.no_grad():
    out_feats = layer(node_feats, adj_matrix)

print("Adjacency matrix", adj_matrix)
print("Input features", node_feats)
print("Output features", out_feats)

Adjacency matrix tensor([[[1., 1., 0., 0.],
         [1., 1., 1., 1.],
         [0., 1., 1., 1.],
         [0., 1., 1., 1.]]])
Input features tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
Output features tensor([[[1., 2.],
         [3., 4.],
         [4., 5.],
         [4., 5.]]])


As we can see, the first node’s output values are the average of itself and the second node. Similarly, we can verify all other nodes.


## GNNs (Graph Neural Networks)
However, in a GNN, we would also want to allow feature exchange between nodes beyond its neighbors. This can be achieved by applying multiple GCN layers, which gives us the final layout of a GNN. The GNN can be build up by a sequence of GCN layers and non-linearities such as ReLU.

![image.png](https://raw.githubusercontent.com/Lightning-AI/tutorials/main/course_UvA-DL/06-graph-neural-networks/gcn_network.png)

Lets test out GNNs with a small problem statement of prediction. It can be done easily with the classic NNs but here we will intoduce graph and how other nodes effect the each others.

# Problem Statement: The "Social Influence" Predictor
Imagine a small social network of 4 people. Some are "Influencers" (Label 1) and some are "Followers" (Label 0).

## The Data:
We only know the "Label" of 2 people. We need the GNN to predict the labels of the other 2 based on who they talk to.

## The Features:
Each person has a simple 2D feature vector (e.g., [Activity Level, Profile Completeness]).

In [6]:
class MultiLayerGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # Layer 1
        self.layer1 = GCNLayer(input_dim, hidden_dim)

        # Layer 2
        self.layer2 = GCNLayer(hidden_dim, hidden_dim)

        # Layer 3
        self.layer3 = GCNLayer(hidden_dim, output_dim)

    def forward(self, x, adj):
        # Layer 1 + Activation
        x = self.layer1(x, adj)
        x = F.relu(x)

        # Layer 2 + Activation
        x = self.layer2(x, adj)
        x = F.relu(x)

        # Layer 3 (No ReLU here, we want output/prediction)
        x = self.layer3(x, adj)
        return x

In [8]:
node_features = torch.tensor([
    [[1.0, 0.5],
     [0.1, 0.2],
      [0.5, 0.9],
      [0.2, 0.1]]
]) # Shape: [1, 4, 2] -> [batch, nodes, features]

adj = torch.tensor([
    [[1, 1, 0, 0],
     [1, 1, 0, 0],
     [0, 0, 1, 1],
     [0, 0, 1, 1]]
]).float()

model = MultiLayerGNN(input_dim=2, hidden_dim=4, output_dim=2)

# Forward Pass
output = model(node_features, adj)

print("(Output):\n", output)

(Output):
 tensor([[[-0.3838,  0.0644],
         [-0.3838,  0.0644],
         [-0.3373,  0.1572],
         [-0.3373,  0.1572]]], grad_fn=<DivBackward0>)


How to use this Output:

we can interpret these numbers like this:


```
Node,      Class 0 Score,          Class 1 Score,          Prediction
0,           -0.3838,                0.0644,                Class 1 (higher score)
1,           -0.3838,                0.0644,                Class 1 (higher score)
2,           -0.3373,                0.1572,                Class 1 (higher score)
3,           -0.3373,                0.1572,                Class 1 (higher score)
```


# Note:
There is one interesting observation , that is if we look closely, in final output feature of *Node 0 and Node 1* & *Node 2 and Node 3* are *similar.*

This tells us that our Multiple Layers successfully mixed the data across the graph. However, because the graph is tiny and the layers are many, the nodes are starting to lose their individual "identity" and are becoming "clones" of their neighbors.

# Now wheather this is good or bad??
Well answer to this question depends upon our use case.
## Community Detection
If we want to group nodes into "clubs" or "communities," `cloning is actually helpful.`

The Logic: If Node 0 and Node 1 are in the same friendship circle, they should have similar embeddings.

The Result: The model ignores tiny individual differences and focuses on the Big Picture. It realizes, "Hey, these four people all belong to the same group," which makes clustering them very easy.
## Node-Level Detection
If we want to tell the difference between two neighbors (e.g., in a Fraud Detection system), `cloning is a disaster.`

The Problem: If Node 0 is a "Normal User" and Node 1 is a "Scammer," but they are friends, a 3-layer GCN will make their features identical.

The Result: Your model will be unable to catch the scammer because the scammer's features have been "washed away" by the normal user's features.

# Now Is there Balance?
## The "Sweet Spot" (The Goldilocks Principle)

In GNN research, we usually try to find a balance. You want nodes to learn from their neighbors, but you don't want them to lose their soul.

```
Feature,                  Low Layers (1-2),                             High Layers (5+)
Information,           Local (Immediate neighbors),                  Global (Entire graph)
Diversity,              High (Nodes stay unique),                  Low (Nodes become ""clones"")
Best For,              Predicting individual traits,                  Predicting community trends
```

